In [ ]:
import html
import json
import logging
import re
import unicodedata
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import polars as pl
import torch
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

logger = logging.getLogger(__name__)

In [ ]:
DATA = Path('../data/Sports')
ITEMS_PATH = DATA / 'items.parquet'
OUTPUT_PATH = DATA / 'enrichment/enriched_v2/summaries.parquet'
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
MODEL_REVISION = 'a09a35458c702b33eeacc393d103063234e8bc28'
BATCH_SIZE = 128
MODEL_DEVICE = 'cuda:0'  # Для проверки без совместимой GPU можно выбрать 'cpu'.
EXPECTED_ITEMS = 18357
VARIANT = 'enriched_v2'

assert not OUTPUT_PATH.exists()
items = pl.read_parquet(ITEMS_PATH).sort('item_id')
original = pl.read_parquet(DATA / 'item_texts.parquet').sort('item_id')
assert items.height == EXPECTED_ITEMS
assert items['item_id'].to_list() == list(range(EXPECTED_ITEMS))
assert items.select('item_id', 'asin').equals(original.select('item_id', 'asin'))
assert items['asin'].n_unique() == EXPECTED_ITEMS

In [ ]:
TAGGED_VARIANTS = frozenset()

NO_DESCRIPTION = 'No description available'

MAX_DESCRIPTION_CHARS = 500

MAX_NEW_TOKENS = 110

MIN_WORDS = 15

MAX_WORDS = 80

MIN_ASCII_SHARE = 0.95

MIN_DISTINCT_SHARE = 0.5

MAX_NGRAM_REPEATS = 3

MAX_CHAR_RUN = 4

STOP_STRINGS = ('</ans>', '\n\n', '### ', '###\n')

_META_PREFIXES = ('sure', 'certainly', 'of course', 'okay', 'ok,', 'here is', "here's", 'here are', 'summary:', 'output:', 'answer:', 'let me', "i'll", 'i will', 'as an ai', 'note:')

_PROMPT_ECHO = ('title:', 'categories:', 'description:', 'hard constraints', '### input')

_EXAMPLE_LEAK = ('wireless earbud', 'crystal-clear audio', 'noise cancellation', 'summary_text')

_MARKDOWN_INLINE = ('**', '__', '`', '|', '•')

_MARKDOWN_LINE_START = re.compile('(?m)^\\s*(?:[#>]+\\s|[-*+]\\s|\\d+\\.\\s)')

def _is_missing(value) -> bool:
    if value is None:
        return True
    if isinstance(value, float):
        return value != value
    if isinstance(value, str):
        return not value.strip()
    return False

def sanitize_text(value, max_len: int | None=None) -> str:
    if _is_missing(value):
        return ''
    text = str(value)
    text = re.sub('<[^>]{0,200}>', ' ', text)
    text = html.unescape(text)
    text = ''.join((' ' if unicodedata.category(ch).startswith('C') else ch for ch in text))
    text = re.sub('\\s+', ' ', text).strip()
    if max_len is not None and len(text) > max_len:
        text = text[:max_len] + '...'
    return text

def categories_to_str(categories) -> str:
    if _is_missing(categories):
        return ''
    flat: list[str] = []
    stack = list(categories) if isinstance(categories, (list, tuple)) else [categories]
    while stack:
        head = stack.pop(0)
        if isinstance(head, (list, tuple)):
            stack = list(head) + stack
        elif not _is_missing(head):
            name = sanitize_text(head)
            if name and name not in flat:
                flat.append(name)
    return ', '.join(flat)

def prepare_item(title=None, categories=None, description=None) -> dict:
    clean_title = sanitize_text(title)
    clean_description = sanitize_text(description, max_len=MAX_DESCRIPTION_CHARS)
    return {'title': clean_title or 'Unknown Product', 'categories_str': categories_to_str(categories), 'description': clean_description or NO_DESCRIPTION, 'empty_input': not clean_title and (not clean_description), 'sparse_input': bool(clean_title) and (not clean_description)}

def build_agg_text(title: str, categories_str: str, description: str) -> str:
    return f'{title} | {categories_str}\n{description}'

def extract_answer(raw: str, variant: str) -> tuple[str | None, str | None]:
    if raw is None:
        return (None, 'empty')
    text = str(raw)
    if variant in TAGGED_VARIANTS:
        if text.count('<ans>') > 1 or text.count('</ans>') > 1:
            return (None, 'multiple_answers')
        if '</ans>' not in text:
            return (None, 'no_close_tag')
        body = text.split('</ans>', 1)[0]
        body = body.split('<ans>', 1)[-1]
    else:
        body = text.split('</ans>', 1)[0]
        body = re.split('\\n\\s*\\n', body, maxsplit=1)[0]
    body = sanitize_text(body)
    if not body:
        return (None, 'empty')
    low = body.lower()
    if any((low.startswith(prefix) for prefix in _META_PREFIXES)):
        return (None, 'meta_preamble')
    return (body, None)

def _words(text: str) -> list[str]:
    return re.findall("[\\w'-]+", text.lower())

def _max_ngram_repeats(words: Sequence[str], n: int=3) -> int:
    if len(words) < n:
        return 0
    grams = Counter((tuple(words[i:i + n]) for i in range(len(words) - n + 1)))
    return max(grams.values())

def check_text(text: str, variant: str) -> str | None:
    low = text.lower()
    if any((marker in low for marker in _PROMPT_ECHO)):
        return 'prompt_echo'
    if any((marker in low for marker in _EXAMPLE_LEAK)):
        return 'example_leak'
    if any((marker in text for marker in _MARKDOWN_INLINE)) or _MARKDOWN_LINE_START.search(text):
        return 'markdown'
    ascii_share = sum((ch.isascii() for ch in text)) / max(len(text), 1)
    if ascii_share < MIN_ASCII_SHARE:
        return 'non_ascii'
    if re.search('(\\w)\\1{%d,}' % MAX_CHAR_RUN, text):
        return 'char_run'
    words = _words(text)
    if not MIN_WORDS <= len(words) <= MAX_WORDS:
        return 'word_count'
    if _max_ngram_repeats(words) >= MAX_NGRAM_REPEATS:
        return 'ngram_repeat'
    if len(set(words)) / len(words) < MIN_DISTINCT_SHARE:
        return 'low_diversity'
    if not text.rstrip().endswith(('.', '!', '?')):
        return 'no_terminal_period'
    return None

def validate(raw: str, variant: str) -> tuple[str | None, str | None]:
    text, reason = extract_answer(raw, variant)
    if reason is not None:
        return (None, reason)
    reason = check_text(text, variant)
    return (None, reason) if reason else (text, None)

@dataclass(frozen=True)
class Attempt:
    temperature: float
    seed: int
    variant_override: str | None = None
    drop_description: bool = False

LADDER: tuple[Attempt, ...] = (Attempt(temperature=0.0, seed=0), Attempt(temperature=0.3, seed=1), Attempt(temperature=0.3, seed=2, variant_override='enriched_v2'), Attempt(temperature=0.0, seed=3, drop_description=True))

In [ ]:
ENRICHED_V2_TEMPLATE = """Product recommendation system - enrich a product description.

Title: {title}
Categories: {categories_str}
Description: {description}

Write a clear one sentence summary describing this product. Start with the brand and product name exactly as given, then its key features, who would use it, and its main benefits. Enrich the input using your knowledge of this product, brand, product line and product category. Add plausible product-specific attributes, likely uses, suitable users and benefits even when they are not stated in the input; reasonable inference and imaginative completion are allowed. Prefer distinctive, concrete details over generic praise. Use plain text only, no formatting or bullet points.

One sentence summary. No thinking. No explanations:"""

def build_prompt(variant: str, title: str, categories_str: str, description: str) -> str:
    if variant != 'enriched_v2':
        raise ValueError(f'unknown prompt variant: {variant!r}')
    return ENRICHED_V2_TEMPLATE.format(title=title, categories_str=categories_str, description=description)

In [ ]:
prepared = []
for row in items.iter_rows(named=True):
    item = prepare_item(row['title'], row['categories'], row['description'])
    item['item_id'] = row['item_id']
    prepared.append(item)

In [ ]:
class Generator:

    def __init__(self, model_id: str, revision: str, device: str) -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.device = torch.device(device)
        if self.device.type == 'cuda' and not torch.cuda.is_available():
            raise RuntimeError("CUDA недоступна: для BACKEND='cuda' нужна NVIDIA GPU")
        if self.device.type == 'cuda' and not torch.cuda.is_bf16_supported():
            raise RuntimeError('Этот запуск настроен на BF16; GPU должна поддерживать BF16')
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, local_files_only=True, padding_side='left')
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(model_id, local_files_only=True, torch_dtype=torch.bfloat16, device_map={'': str(self.device)}).eval()
        self.revision = revision
        logger.info('Device: %s; модель: %s; revision: %s', str(self.device), model_id, self.revision)

    def generate(self, prompts: list[str], *, temperature: float, seed: int, max_tokens: int) -> list[str]:
        import torch
        from transformers import set_seed
        set_seed(seed)
        texts = []
        for prompt in prompts:
            prefill = '<ans>' if prompt.rstrip().endswith('<ans>') else ''
            if prefill:
                prompt = prompt.rstrip()[:-len(prefill)].rstrip()
            texts.append(self.tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True) + prefill)
        inputs = self.tokenizer(texts, padding=True, return_tensors='pt', add_special_tokens=False).to(self.device)
        sampling = {'temperature': temperature, 'top_p': 1.0, 'top_k': 0} if temperature > 0 else {}
        with torch.inference_mode():
            generated = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=temperature > 0, repetition_penalty=1.1, stop_strings=list(STOP_STRINGS), tokenizer=self.tokenizer, pad_token_id=self.tokenizer.pad_token_id, **sampling)
        return self.tokenizer.batch_decode(generated[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [ ]:
def enrich_batch(batch: list[dict], variant: str, generator) -> list[dict]:
    records = [{'item_id': item['item_id'], 'variant': variant, 'attempts': 0, 'reject_reasons': [], 'attempts_log': [], 'summary': None, 'status': 'pending', 'fallback': False, 'fallback_reason': None, 'raw': None} for item in batch]
    pending = [index for index, item in enumerate(batch) if not item['empty_input']]
    for attempt in LADDER:
        if not pending:
            break
        effective = attempt.variant_override or variant
        prompts = [build_prompt(effective, batch[index]['title'], batch[index]['categories_str'], NO_DESCRIPTION if attempt.drop_description else batch[index]['description']) for index in pending]
        outputs = generator.generate(prompts, temperature=attempt.temperature, seed=attempt.seed, max_tokens=MAX_NEW_TOKENS)
        if len(outputs) != len(pending):
            raise RuntimeError('Генератор вернул неверное число ответов')
        retry = []
        for index, raw in zip(pending, outputs, strict=True):
            record = records[index]
            record['attempts'] += 1
            text, reason = validate(raw, effective)
            record['raw'] = raw
            record['attempts_log'].append({'variant': effective, 'temperature': attempt.temperature, 'reason': reason, 'text': str(raw)[:300]})
            if reason is None:
                record.update(summary=text, status='ok')
            else:
                record['reject_reasons'].append(reason)
                retry.append(index)
        pending = retry
    for item, record in zip(batch, records, strict=True):
        if record['summary'] is None:
            record.update(summary=build_agg_text(item['title'], item['categories_str'], item['description']), status='fallback', fallback=True, fallback_reason='empty_input' if item['empty_input'] else 'invalid_after_retries')
        record['sparse_input'] = bool(item['sparse_input'])
    return records

In [ ]:
snapshot = snapshot_download(MODEL_ID, revision=MODEL_REVISION,
                             allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model'])
generator = Generator(snapshot, MODEL_REVISION, MODEL_DEVICE)
records = []
for start in tqdm(range(0, len(prepared), BATCH_SIZE)):
    records.extend(enrich_batch(prepared[start:start + BATCH_SIZE], VARIANT, generator))

In [ ]:
summaries = pl.DataFrame([{
    'item_id': r['item_id'], 'text': r['summary'], 'variant': r['variant'],
    'status': r['status'], 'fallback': r['fallback'], 'attempts': r['attempts'],
} for r in records]).join(items.select('item_id', 'asin'), on='item_id', validate='1:1').sort('item_id')
assert summaries.height == EXPECTED_ITEMS
assert summaries.select('item_id', 'asin').equals(items.select('item_id', 'asin'))
assert summaries['text'].str.len_chars().min() > 0
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
assert not OUTPUT_PATH.exists()
summaries.write_parquet(OUTPUT_PATH)
summaries.head()